# Yasiris Andrea Mercado Reales 
# Laboratorio 2 — Árbol de Merkle (Merkle Tree)

**Implementación en Python con SHA-256**

Este notebook implementa un Árbol de Merkle desde cero y ejecuta el experimento solicitado:

1. Crear 5 transacciones simuladas.
2. Construir el árbol y mostrar la raíz (Merkle Root).
3. Modificar una transacción y demostrar que la raíz cambia.
4. Generar una prueba de inclusión (Merkle Proof) para la transacción 3 y verificarla.
5. Verificar con un dato incorrecto → debe fallar.

**Reglas de construcción:**
- Cada hoja contiene el hash SHA-256 de un bloque de datos.
- Cada nodo interno contiene el hash SHA-256 de la concatenación de sus dos hijos.
- Si el número de hojas o ramas en un nivel es impar, la última se duplica.
- La raíz (Merkle Root) es el hash que representa todo el conjunto.


## 1. Implementación del Árbol de Merkle

In [ ]:
import hashlib


def sha256(data: str) -> str:
    """Devuelve el hash SHA-256 (hexadecimal) de una cadena de texto."""
    return hashlib.sha256(data.encode("utf-8")).hexdigest()


class MerkleTree:
    def __init__(self, transactions):
        if not transactions:
            raise ValueError("Se necesita al menos una transacción para construir el árbol.")
        self.transactions = list(transactions)
        self.levels = []  # levels[0] = hojas, levels[-1] = [raíz]
        self._build_tree()

    # ------------------------------------------------------------
    # Construcción del árbol
    # ------------------------------------------------------------
    def _build_tree(self):
        # Nivel 0: hash de cada transacción (hojas)
        leaves = [sha256(tx) for tx in self.transactions]
        self.levels = [leaves]

        current_level = leaves
        while len(current_level) > 1:
            # Si el nivel es impar, se duplica el último elemento
            if len(current_level) % 2 == 1:
                current_level = current_level + [current_level[-1]]

            next_level = []
            for i in range(0, len(current_level), 2):
                left = current_level[i]
                right = current_level[i + 1]
                parent_hash = sha256(left + right)
                next_level.append(parent_hash)

            self.levels.append(next_level)
            current_level = next_level

    @property
    def root(self):
        """Merkle Root: hash raíz que representa todo el conjunto de datos."""
        return self.levels[-1][0]

    # ------------------------------------------------------------
    # Prueba de inclusión (Merkle Proof)
    # ------------------------------------------------------------
    def get_proof(self, index):
        """Genera la prueba de inclusión para la hoja en la posición `index`."""
        if index < 0 or index >= len(self.transactions):
            raise IndexError("Índice de transacción fuera de rango.")

        proof = []
        idx = index

        for level in self.levels[:-1]:
            level_padded = level[:]
            if len(level_padded) % 2 == 1:
                level_padded.append(level_padded[-1])

            is_right_node = (idx % 2 == 1)
            sibling_index = idx - 1 if is_right_node else idx + 1
            sibling_hash = level_padded[sibling_index]

            position = "left" if is_right_node else "right"
            proof.append((sibling_hash, position))

            idx = idx // 2

        return proof

    # ------------------------------------------------------------
    # Verificación de una prueba de inclusión
    # ------------------------------------------------------------
    @staticmethod
    def verify_proof(data, proof, root):
        """Verifica que `data` pertenece al árbol cuya raíz es `root`."""
        computed_hash = sha256(data)

        for sibling_hash, position in proof:
            if position == "right":
                computed_hash = sha256(computed_hash + sibling_hash)
            else:
                computed_hash = sha256(sibling_hash + computed_hash)

        return computed_hash == root

    # ------------------------------------------------------------
    # Utilidades de visualización
    # ------------------------------------------------------------
    def print_tree(self):
        """Imprime el árbol nivel por nivel (raíz al final)."""
        for i, level in enumerate(self.levels):
            label = "Raíz (Merkle Root)" if i == len(self.levels) - 1 else f"Nivel {i}"
            print(f"\n{label}:")
            for h in level:
                print(f"  {h}")

print("Clase MerkleTree definida correctamente.")


Clase MerkleTree definida correctamente.


## 2. Crear 5 transacciones simuladas

In [ ]:
transacciones = [
    "TX1: Ana paga 50 a Bruno",
    "TX2: Bruno paga 20 a Carla",
    "TX3: Carla paga 100 a Diego",
    "TX4: Diego paga 15 a Elena",
    "TX5: Elena paga 30 a Ana",
]

for i, tx in enumerate(transacciones):
    print(f"[{i}] {tx}")


[0] TX1: Ana paga 50 a Bruno
[1] TX2: Bruno paga 20 a Carla
[2] TX3: Carla paga 100 a Diego
[3] TX4: Diego paga 15 a Elena
[4] TX5: Elena paga 30 a Ana


## 3. Construir el árbol y mostrar la raíz (Merkle Root)

In [ ]:
arbol = MerkleTree(transacciones)
arbol.print_tree()

print(f"\n>>> MERKLE ROOT: {arbol.root}")



Nivel 0:
  21af48478385ea935abefc6974bf94afcc439fa4ed9cc31a1ebb6b0a25c54d59
  a55c56fc73aa9a055cc6ebc6f4f46b89cdae2afb91b5e4a690209792b10b9b39
  89c914d51046a793cc293a934aee6197331a52348891082f56c7effb5e5583c9
  b624272e2663395a2d36d5a30e82060bb5ed1cc86d6f0884d82cecef418f23d1
  6ae4711dba4f3098594cc68648172357b6a73592e5719f28ece2041b15ad2014

Nivel 1:
  65de03ee2ad4593485dec5d1a27c939ee32af9b174117c329895a68886a3892f
  3d5d2857251527737f4e03ab4d895eca9ad4874438d9d0998e40deb124e75d36
  fb40366a16047b1fd722e2a273e65dfcdc8b1b08042e29b676b9a88160abd9c3

Nivel 2:
  813b205d1460b4f843a17a2230c481e234c6a7578dd89bb5737e63d63235b778
  fa9c3fe2724912aed9b29b687b11dd0633b9e02e7014d0890a516794a9562ae5

Raíz (Merkle Root):
  3589b5bb7945794e0d4687d167d1be9a787f7faf0f08089c70e86f089c262a93

>>> MERKLE ROOT: 3589b5bb7945794e0d4687d167d1be9a787f7faf0f08089c70e86f089c262a93


## 4. Diagrama del árbol (ASCII)

Con 5 hojas, el Nivel 1 queda con 3 nodos (impar) → se duplica el último para calcular el Nivel 2.


In [ ]:
def diagrama_ascii(arbol):
    L0 = arbol.levels[0]
    L1 = arbol.levels[1]
    L2 = arbol.levels[2]
    root = arbol.root

    def corto(h):
        return h[:8] + ".."

    print("                                   RAÍZ (Merkle Root)")
    print(f"                                   [{corto(root)}]")
    print("                                  /                \\")
    print(f"                        [{corto(L2[0])}]         [{corto(L2[1])}]")
    print("                        /          \\                 \\")
    print(f"              [{corto(L1[0])}]  [{corto(L1[1])}]      [{corto(L1[2])}]")
    print("               /      \\      /      \\         (dup) /")
    print(f"        [{corto(L0[0])}][{corto(L0[1])}][{corto(L0[2])}][{corto(L0[3])}]  [{corto(L0[4])}]")
    print("          TX1      TX2      TX3      TX4         TX5")
    print()
    print("Nota: como el Nivel 1 tiene 3 nodos (impar), el 3er nodo se duplica")
    print("      consigo mismo para calcular el hash del Nivel 2 (rama derecha).")

diagrama_ascii(arbol)


                                   RAÍZ (Merkle Root)
                                   [3589b5bb..]
                                  /                \
                        [813b205d..]         [fa9c3fe2..]
                        /          \                 \
              [65de03ee..]  [3d5d2857..]      [fb40366a..]
               /      \      /      \         (dup) /
        [21af4847..][a55c56fc..][89c914d5..][b624272e..]  [6ae4711d..]
          TX1      TX2      TX3      TX4         TX5

Nota: como el Nivel 1 tiene 3 nodos (impar), el 3er nodo se duplica
      consigo mismo para calcular el hash del Nivel 2 (rama derecha).


## 5. Modificar una transacción y demostrar que la raíz cambia

In [ ]:
transacciones_modificadas = transacciones.copy()
original_tx3 = transacciones_modificadas[2]
transacciones_modificadas[2] = "TX3: Carla paga 999999 a Diego"  # dato alterado

print(f"Transacción original  [2]: {original_tx3}")
print(f"Transacción modificada[2]: {transacciones_modificadas[2]}")

arbol_modificado = MerkleTree(transacciones_modificadas)

print(f"\n>>> MERKLE ROOT ORIGINAL:   {arbol.root}")
print(f">>> MERKLE ROOT MODIFICADO: {arbol_modificado.root}")

if arbol.root != arbol_modificado.root:
    print("\n✅ La raíz CAMBIÓ tras modificar una sola transacción, como se esperaba.")
else:
    print("\n❌ ERROR: la raíz no cambió (no debería ocurrir).")


Transacción original  [2]: TX3: Carla paga 100 a Diego
Transacción modificada[2]: TX3: Carla paga 999999 a Diego

>>> MERKLE ROOT ORIGINAL:   3589b5bb7945794e0d4687d167d1be9a787f7faf0f08089c70e86f089c262a93
>>> MERKLE ROOT MODIFICADO: f7f9ddd22d0dc11e08450458de4512d82d8e6190dd8582c8f923d5c35d35e298

✅ La raíz CAMBIÓ tras modificar una sola transacción, como se esperaba.


## 6. Prueba de inclusión (Merkle Proof) para la transacción 3

In [ ]:
indice_tx3 = 2  # TX3 está en el índice 2
prueba_tx3 = arbol.get_proof(indice_tx3)

print(f"Transacción a probar: \'{transacciones[indice_tx3]}\'\n")
print("Prueba de inclusión (hash hermano, posición):")
for h, pos in prueba_tx3:
    print(f"  - {h}  ({pos})")


Transacción a probar: 'TX3: Carla paga 100 a Diego'

Prueba de inclusión (hash hermano, posición):
  - b624272e2663395a2d36d5a30e82060bb5ed1cc86d6f0884d82cecef418f23d1  (right)
  - 65de03ee2ad4593485dec5d1a27c939ee32af9b174117c329895a68886a3892f  (left)
  - fa9c3fe2724912aed9b29b687b11dd0633b9e02e7014d0890a516794a9562ae5  (right)


### 6.1 Verificación con el dato CORRECTO → debe ser válida

In [ ]:
es_valida = MerkleTree.verify_proof(transacciones[indice_tx3], prueba_tx3, arbol.root)

print(f"Dato verificado: \'{transacciones[indice_tx3]}\'")
print(f"Raíz esperada:   {arbol.root}")
print(f"\nResultado de la verificación -> {"VÁLIDA ✅" if es_valida else "INVÁLIDA ❌"}")

assert es_valida, "La prueba debería ser válida"


Dato verificado: 'TX3: Carla paga 100 a Diego'
Raíz esperada:   3589b5bb7945794e0d4687d167d1be9a787f7faf0f08089c70e86f089c262a93

Resultado de la verificación -> VÁLIDA ✅


## 7. Verificación con un dato incorrecto → debe fallar

In [ ]:
dato_falso = "TX3: Carla paga 100000 a Diego (dato alterado por un atacante)"

es_valida_falsa = MerkleTree.verify_proof(dato_falso, prueba_tx3, arbol.root)

print(f"Dato falso usado: \'{dato_falso}\'")
print(f"Raíz esperada:    {arbol.root}")
print(f"\nResultado de la verificación -> {"VÁLIDA ✅" if es_valida_falsa else "INVÁLIDA ❌"}")

assert not es_valida_falsa, "La prueba con dato falso NO debería ser válida"
print("\n✅ Como se esperaba, la verificación con datos alterados FALLA.")


Dato falso usado: 'TX3: Carla paga 100000 a Diego (dato alterado por un atacante)'
Raíz esperada:    3589b5bb7945794e0d4687d167d1be9a787f7faf0f08089c70e86f089c262a93

Resultado de la verificación -> INVÁLIDA ❌

✅ Como se esperaba, la verificación con datos alterados FALLA.


## 8. Conclusión

- La **Merkle Root** cambia de forma completa e impredecible ante la modificación de **una sola transacción** (efecto avalancha de SHA-256).
- Una **prueba de inclusión** (Merkle Proof) permite verificar que una transacción específica pertenece al árbol **sin necesidad de conocer todas las demás transacciones**, solo se necesitan `log2(n)` hashes hermanos.
- Si el dato que se intenta verificar no coincide exactamente con el original, el hash recalculado en la raíz **no coincide** con la Merkle Root real, y la verificación falla correctamente.


## 9. Declaración de uso de IA generativa

Conforme al código de honor del curso, se declara explícitamente el uso de IA
generativa (Claude, Anthropic) en este entregable:

- **Qué se usó de la IA:** Para generar una primero version del codigo base clase merkletree, script de experimentos a partir de las expecificaciones del alboratorio, generación de imagenes y diagrama. resolución de dudas puntuales y correcion de partes del codigo
- **Qué no vino de la IA:** los datos de las 5 transacciones simuladas y depuración del codigo y revisión definición y verificación del comporatmiento del arbol con datos propios.
- **Responsabilidad:** el estudiante ha revisado, ejecutado y comprende algunas  partes
  del código entregado (construcción del árbol, cálculo de la Merkle Root,
  generación y verificación de la prueba de inclusión) y puede explicar cualquier
  elemento de la entrega.
